# Router Engine

In [1]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "httpx<0.28.0",
    "openai==1.40.0",
    "llama-index==0.10.68",
    "llama-index-llms-openai==0.1.31",
    "llama-index-embeddings-openai==0.1.11",
    "python-dotenv==1.0.0",
    "nest_asyncio",
    "--quiet"
])
print("Done")

Done


## Setup

In [2]:
from helper import get_openai_api_key

OPENAI_API_KEY = get_openai_api_key()

In [3]:
import nest_asyncio

nest_asyncio.apply()

## Load Data

In [23]:
from llama_index.core import SimpleDirectoryReader

# load documents
documents = SimpleDirectoryReader(input_files=["/home/as/Documents/GitHub/Llamaindex_Practice/metagpt.pdf"]).load_data()

## Define LLM and Embedding model

In [5]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [12]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-4o-mini")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

## Define Summary Index and Vector Index over the Same Data

In [14]:
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)

## Define Query Engines and Set Metadata

In [15]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)
vector_query_engine = vector_index.as_query_engine()

In [16]:
from llama_index.core.tools import QueryEngineTool


summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=(
        "Useful for summarization questions related to MetaGPT"
    ),
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=(
        "Useful for retrieving specific context from the MetaGPT paper."
    ),
)

## Define Router Query Engine

In [17]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector


query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True
)

In [18]:
response = query_engine.query("What is the summary of the document?")
print(str(response))

Selecting query engine 0: The question asks for a summary of the document, which aligns with the purpose of choice 1, as it is useful for summarization questions..
The document introduces MetaGPT, a meta-programming framework designed for multi-agent collaboration using Large Language Models (LLMs). It addresses the challenges faced by existing LLM-based systems in solving complex tasks due to issues like logic inconsistencies and cascading hallucinations. MetaGPT incorporates Standardized Operating Procedures (SOPs) to streamline workflows, allowing agents with specialized roles to collaborate effectively. The framework utilizes structured communication and an assembly line approach to break down complex tasks into manageable subtasks.

MetaGPT demonstrates significant improvements in code generation quality and task completion rates compared to previous frameworks, achieving state-of-the-art performance on benchmarks like HumanEval and MBPP. The framework includes a novel executable 

In [19]:
print(len(response.source_nodes))

34


In [20]:
response = query_engine.query(
    "How do agents share information with other agents?"
)
print(str(response))

Selecting query engine 1: The question asks for specific information about how agents share information, which aligns with retrieving specific context from the MetaGPT paper..
Agents share information with other agents through a shared message pool. This mechanism allows agents to publish structured messages and access messages from other entities directly. By utilizing this shared pool, agents can retrieve the necessary information without needing to inquire about other agents, thereby enhancing communication efficiency. Additionally, agents can subscribe to relevant messages based on their role profiles, ensuring they receive only task-related information and minimizing distractions from irrelevant details.


## put everything together

In [21]:
from utils import get_router_query_engine

query_engine = get_router_query_engine("metagpt.pdf")

In [22]:
response = query_engine.query("Tell me about the ablation study results?")
print(str(response))

Selecting query engine 1: The question asks for specific results from an ablation study, which would require retrieving specific context from the MetaGPT paper..
The provided information does not include details about the ablation study results. Therefore, I cannot provide any information on that topic.
